In [22]:
import numpy as np
import pandas as pd

In [23]:
df = pd.read_csv('data/kacha_river_2023.csv', parse_dates=['datetime'])
df_test = pd.read_csv('data/kacha_river_station_1_2023.csv', parse_dates=['datetime'])
df.head()

,datetime,temp_c,precip_mm,humidity_pct,wind_speed_mph,pressure_hpa,water_level_cm
0,2023-01-01 00:00:00,-3.1,0.0,71.1,11.0,1020.4,80.4
1,2023-01-01 06:00:00,-5.0,0.0,78.6,12.3,1026.8,80.3
2,2023-01-01 12:00:00,-2.6,0.2,70.2,15.0,1021.6,79.0
3,2023-01-01 18:00:00,0.0,0.0,60.7,12.8,1011.7,79.5
4,2023-01-02 00:00:00,-5.3,0.0,73.9,11.6,1018.5,78.5


In [24]:
df['water_level_trend_3d'] = df['water_level_cm'] - df['water_level_cm'].shift(12)
df['target_6h'] = df['water_level_cm'].shift(-1)
df['target_24h'] = df['water_level_cm'].shift(-4)
df['target_72h'] = df['water_level_cm'].shift(-12)

df_test['water_level_trend_3d'] = df_test['water_level_cm'] - df_test['water_level_cm'].shift(12)
df_test['target_6h'] = df_test['water_level_cm'].shift(-1)
df_test['target_24h'] = df_test['water_level_cm'].shift(-4)
df_test['target_72h'] = df_test['water_level_cm'].shift(-12)

df = df.dropna().reset_index(drop=True)
df_test = df_test.dropna().reset_index(drop=True)
df.head()

,datetime,temp_c,precip_mm,humidity_pct,wind_speed_mph,pressure_hpa,water_level_cm,water_level_trend_3d,target_6h,target_24h,target_72h
0,2023-01-04 00:00:00,-4.0,0.0,74.3,12.9,1017.3,79.9,-0.5,80.8,82.2,84.7
1,2023-01-04 06:00:00,-10.5,0.0,76.3,7.3,1009.6,80.8,0.5,80.4,83.3,85.6
2,2023-01-04 12:00:00,-9.9,0.0,79.5,11.4,1021.6,80.4,1.4,79.1,83.7,82.3
3,2023-01-04 18:00:00,-6.4,0.0,74.0,15.5,1016.9,79.1,-0.4,82.2,84.8,81.2
4,2023-01-05 00:00:00,-7.8,0.9,71.5,13.8,1009.3,82.2,3.7,83.3,80.9,80.6


In [25]:
def check_nan(df):
    print("Проверка на пропуски")
    miss = df.isnull().sum()
    missing_val = (miss / len(df))
    missing_df = pd.DataFrame({
        'Количество пропусков': miss,
        'Доля от всех данных': missing_val
    })
    print(missing_df)

check_nan(df)

Проверка на пропуски
                      Количество пропусков  Доля от всех данных
datetime                                 0                  0.0
temp_c                                   0                  0.0
precip_mm                                0                  0.0
humidity_pct                             0                  0.0
wind_speed_mph                           0                  0.0
pressure_hpa                             0                  0.0
water_level_cm                           0                  0.0
water_level_trend_3d                     0                  0.0
target_6h                                0                  0.0
target_24h                               0                  0.0
target_72h                               0                  0.0


In [26]:
from sklearn.preprocessing import StandardScaler

y_6_hour = df['target_6h']
y_24_hour = df['target_24h']
y_72_hour = df['target_72h']

date_test = df_test['datetime']
y_6_hour_test = df_test['target_6h']
y_24_hour_test = df_test['target_24h']
y_72_hour_test = df_test['target_72h']

df = df.drop(columns=['datetime', 'target_6h', 'target_24h', 'target_72h'])
df_test = df_test.drop(columns=['datetime', 'target_6h', 'target_24h', 'target_72h'])

X_train, X_val = df, df_test
y_6_hour_train, y_6_hour_val = y_6_hour, y_6_hour_test
y_24_hour_train, y_24_hour_val = y_24_hour, y_24_hour_test
y_72_hour_train, y_72_hour_val = y_72_hour, y_72_hour_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_val)


In [27]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import xgboost as xgb

xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 7, 8],
    'learning_rate': [0.01, 0.2, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.8, 0.85, 1.0]
}

tscv = TimeSeriesSplit(n_splits=3)

grid_search_6 = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

grid_search_24 = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

grid_search_72 = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

In [28]:
grid_search_6.fit(X_train_scaled, y_6_hour_train)
best_model_6 = grid_search_6.best_estimator_

grid_search_24.fit(X_train_scaled, y_24_hour_train)
best_model_24 = grid_search_24.best_estimator_

grid_search_72.fit(X_train_scaled, y_72_hour_train)
best_model_72 = grid_search_72.best_estimator_


y_pred_6 = best_model_6.predict(X_test_scaled)
y_pred_24 = best_model_24.predict(X_test_scaled)
y_pred_72 = best_model_72.predict(X_test_scaled)

Fitting 3 folds for each of 540 candidates, totalling 1620 fits
Fitting 3 folds for each of 540 candidates, totalling 1620 fits
Fitting 3 folds for each of 540 candidates, totalling 1620 fits


In [29]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")

print('Метрики для прогноза на 6 часов вперед:')
metrics(y_6_hour_val, y_pred_6)

print('\nМетрики для прогноза на 24 часа вперед:')
metrics(y_24_hour_val, y_pred_24)

print('\nМетрики для прогноза на 72 часа вперед:')
metrics(y_72_hour_val, y_pred_72)

Метрики для прогноза на 6 часов вперед:
MAE: 4.4195
RMSE: 6.0066

Метрики для прогноза на 24 часа вперед:
MAE: 4.4146
RMSE: 6.2372

Метрики для прогноза на 72 часа вперед:
MAE: 5.5464
RMSE: 7.8565


In [30]:
def classify_flood_level(water_level):
    if water_level < 100:
        return "норма"
    elif water_level < 120:
        return "предупреждение"
    else:
        return "паводок"

classify_6 = [classify_flood_level(x) for x in y_pred_6]
classify_24 = [classify_flood_level(x) for x in y_pred_24]
classify_72 = [classify_flood_level(x) for x in y_pred_72]

In [31]:
finally_data = pd.DataFrame({
    'date': date_test,
    'predict_6_hour': y_pred_6,
    'predict_24_hour': y_pred_24,
    'predict_72_hour': y_pred_72,
    'classify_6_hour': classify_6,
    'classify_24_hour': classify_24,
    'classify_72_hour': classify_72,
})

finally_data.to_csv('predictions.csv', index=False)